# Data Science Case Study 2: Automatic Feature Engineering with GAs

## 🎯 Objetivo

Usar Algoritmos Genéticos para **crear features automáticamente** mediante:
1. Combinaciones de features existentes
2. Transformaciones matemáticas
3. Interacciones entre variables
4. Selección de las mejores features generadas

## 📚 Lo que Aprenderás

- Feature engineering automático vs manual
- Codificación de transformaciones
- Evaluación de features generadas
- Prevenir overfitting en feature creation
- Interpretabilidad de features engineered

## 🔧 Caso Real

**Problema:** Predicción de diabetes
- Features básicas: edad, BMI, presión, glucosa, etc.
- **Objetivo:** Crear features derivadas que mejoren predicción
- **Ejemplos:** BMI², edad×glucosa, ratios, transformaciones log, etc.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import sys

sys.path.append('../GA_Tutorial_1_Basics')
sys.path.append('../GA_Tutorial_2_Intermediate')
from ga_utils_basics import *
from ga_utils_intermediate import *

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Imports successful!")

## 1. Cargar Datos

In [ ]:
# Cargar diabetes dataset
data = load_diabetes()
X, y = data.data, data.target
feature_names = data.feature_names

print("="*70)
print("DATASET: Diabetes Progression Prediction")
print("="*70)
print(f"Samples: {X.shape[0]}")
print(f"Original features: {X.shape[1]}")
print(f"Feature names: {feature_names}")
print(f"Target: Disease progression after one year")

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Escalar
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Baseline
baseline_model = Ridge()
baseline_model.fit(X_train_scaled, y_train)
baseline_score = r2_score(y_test, baseline_model.predict(X_test_scaled))

print(f"\nBaseline R² (features originales): {baseline_score:.4f}")
print(f"\n💡 Objetivo: Mejorar R² mediante feature engineering automático")

## 2. Definir Espacio de Transformaciones

### Transformaciones Posibles:
1. **Unarias:** x², √x, log(x), exp(x), |x|, 1/x
2. **Binarias:** x+y, x-y, x×y, x/y, x^y
3. **Agregadas:** max(x,y), min(x,y)

### Codificación del Cromosoma:
```
Cada feature generada se codifica como:
[type, operation, feature1_idx, feature2_idx, include]
```

In [ ]:
# Definir transformaciones
UNARY_OPS = {
    'square': lambda x: x ** 2,
    'sqrt': lambda x: np.sqrt(np.abs(x)),
    'log': lambda x: np.log1p(np.abs(x)),
    'exp': lambda x: np.clip(np.exp(x), -1e10, 1e10),
    'abs': lambda x: np.abs(x),
    'inv': lambda x: 1 / (x + 1e-10)
}

BINARY_OPS = {
    'add': lambda x, y: x + y,
    'subtract': lambda x, y: x - y,
    'multiply': lambda x, y: x * y,
    'divide': lambda x, y: x / (y + 1e-10),
    'max': lambda x, y: np.maximum(x, y),
    'min': lambda x, y: np.minimum(x, y)
}

print(f"Transformaciones unarias: {list(UNARY_OPS.keys())}")
print(f"Transformaciones binarias: {list(BINARY_OPS.keys())}")
print(f"\nEspacio total posible:")
print(f"  Unarias: {len(UNARY_OPS)} ops × {X.shape[1]} features = {len(UNARY_OPS) * X.shape[1]} features")
print(f"  Binarias: {len(BINARY_OPS)} ops × {X.shape[1]*(X.shape[1]-1)//2} pairs = {len(BINARY_OPS) * X.shape[1]*(X.shape[1]-1)//2} features")
print(f"  Total posible: {len(UNARY_OPS) * X.shape[1] + len(BINARY_OPS) * X.shape[1]*(X.shape[1]-1)//2} features")
print(f"\n💡 GA nos ayudará a encontrar las mejores!")

## 3. Implementar Feature Engineering con GA

In [ ]:
def decode_feature_engineering_chromosome(chromosome, n_original_features, max_new_features=20):
    """
    Decodifica cromosoma a conjunto de features engineered.
    
    Cromosoma: array de valores [0,1]
    Cada 5 genes codifican una feature:
    [type, op_idx, feat1, feat2, include]
    """
    genes_per_feature = 5
    n_encoded = len(chromosome) // genes_per_feature
    
    features_spec = []
    
    for i in range(min(n_encoded, max_new_features)):
        start = i * genes_per_feature
        genes = chromosome[start:start+genes_per_feature]
        
        # Decodificar
        is_binary = genes[0] > 0.5
        
        if is_binary:
            op_idx = int(genes[1] * len(BINARY_OPS)) % len(BINARY_OPS)
            op_name = list(BINARY_OPS.keys())[op_idx]
            feat1_idx = int(genes[2] * n_original_features) % n_original_features
            feat2_idx = int(genes[3] * n_original_features) % n_original_features
            include = genes[4] > 0.5
            
            if include and feat1_idx != feat2_idx:
                features_spec.append({
                    'type': 'binary',
                    'operation': op_name,
                    'feature1': feat1_idx,
                    'feature2': feat2_idx
                })
        else:
            op_idx = int(genes[1] * len(UNARY_OPS)) % len(UNARY_OPS)
            op_name = list(UNARY_OPS.keys())[op_idx]
            feat_idx = int(genes[2] * n_original_features) % n_original_features
            include = genes[4] > 0.5
            
            if include:
                features_spec.append({
                    'type': 'unary',
                    'operation': op_name,
                    'feature': feat_idx
                })
    
    return features_spec


def create_engineered_features(X, features_spec, feature_names):
    """
    Crea features basadas en especificación.
    """
    new_features = []
    new_feature_names = []
    
    for spec in features_spec:
        try:
            if spec['type'] == 'unary':
                op = UNARY_OPS[spec['operation']]
                feat = X[:, spec['feature']]
                new_feat = op(feat)
                
                # Check for NaN/Inf
                if np.all(np.isfinite(new_feat)):
                    new_features.append(new_feat)
                    new_feature_names.append(f"{spec['operation']}({feature_names[spec['feature']]})")
                    
            else:  # binary
                op = BINARY_OPS[spec['operation']]
                feat1 = X[:, spec['feature1']]
                feat2 = X[:, spec['feature2']]
                new_feat = op(feat1, feat2)
                
                if np.all(np.isfinite(new_feat)):
                    new_features.append(new_feat)
                    fname1 = feature_names[spec['feature1']]
                    fname2 = feature_names[spec['feature2']]
                    new_feature_names.append(f"{fname1}_{spec['operation']}_{fname2}")
        except:
            pass
    
    if len(new_features) > 0:
        return np.column_stack(new_features), new_feature_names
    else:
        return np.zeros((X.shape[0], 1)), ['dummy']


def evaluate_feature_engineering(chromosome, X_train, y_train, n_features, cv=3):
    """
    Evalúa calidad de features engineered.
    """
    try:
        # Decodificar
        features_spec = decode_feature_engineering_chromosome(chromosome, n_features)
        
        if len(features_spec) == 0:
            return 0.0
        
        # Crear features
        new_features, _ = create_engineered_features(X_train, features_spec, feature_names)
        
        # Combinar con originales
        X_combined = np.hstack([X_train, new_features])
        
        # Evaluar
        model = Ridge(alpha=1.0)
        scores = cross_val_score(model, X_combined, y_train, cv=cv, scoring='r2')
        
        # Penalizar por muchas features (overfitting)
        complexity_penalty = len(features_spec) * 0.002
        
        fitness = np.mean(scores) - complexity_penalty
        
        return max(fitness, 0.0)
    except:
        return 0.0

print("✓ Feature Engineering funciones definidas")

## 4. Ejecutar GA para Feature Engineering

In [ ]:
def feature_engineering_ga(X_train, y_train, pop_size=30, max_generations=40):
    """
    GA para feature engineering automático.
    """
    n_features = X_train.shape[1]
    chromosome_length = 5 * 20  # 20 posibles features engineered
    
    population = initialize_population_real(pop_size, chromosome_length, (0, 1))
    
    history = {'best_fitness': [], 'avg_fitness': [], 'n_features': []}
    
    print("="*70)
    print("FEATURE ENGINEERING GA")
    print("="*70)
    print(f"Original features: {n_features}")
    print(f"Max new features: 20")
    print(f"Population: {pop_size}, Generations: {max_generations}")
    print("\nEvolution:")
    print("-"*70)
    
    for gen in range(max_generations):
        # Evaluar
        fitness_values = np.array([
            evaluate_feature_engineering(ind, X_train, y_train, n_features)
            for ind in population
        ])
        
        best_idx = np.argmax(fitness_values)
        best_chromosome = population[best_idx]
        
        features_spec = decode_feature_engineering_chromosome(best_chromosome, n_features)
        
        history['best_fitness'].append(np.max(fitness_values))
        history['avg_fitness'].append(np.mean(fitness_values))
        history['n_features'].append(len(features_spec))
        
        if gen % 10 == 0 or gen == max_generations - 1:
            print(f"Gen {gen:2d} | Best R²: {np.max(fitness_values):.4f} | "
                  f"Avg: {np.mean(fitness_values):.4f} | New features: {len(features_spec):2d}")
        
        # Evolution
        parents = rank_based_selection(population, fitness_values, pop_size)
        
        offspring = []
        for i in range(0, pop_size, 2):
            c1, c2 = simulated_binary_crossover(parents[i], parents[min(i+1, pop_size-1)], eta=20, bounds=(0,1))
            c1 = polynomial_mutation(c1, 0.15, (0,1), eta=20)
            c2 = polynomial_mutation(c2, 0.15, (0,1), eta=20)
            offspring.extend([c1, c2])
        
        population = np.array(offspring[:pop_size])
        population[0] = best_chromosome
    
    print("-"*70)
    print("✓ Feature Engineering completed")
    
    return best_chromosome, history

# Ejecutar
best_chromosome, history = feature_engineering_ga(
    X_train_scaled, y_train,
    pop_size=30,
    max_generations=40
)

## 5. Analizar Features Generadas

In [ ]:
# Decodificar mejores features
best_features_spec = decode_feature_engineering_chromosome(best_chromosome, X_train_scaled.shape[1])

print("="*70)
print("FEATURES ENGINEERED ENCONTRADAS")
print("="*70)
print(f"\nTotal: {len(best_features_spec)} new features")
print("\nDetalle:")

for i, spec in enumerate(best_features_spec):
    if spec['type'] == 'unary':
        print(f"{i+1}. {spec['operation']}({feature_names[spec['feature']]})")
    else:
        print(f"{i+1}. {feature_names[spec['feature1']]} {spec['operation']} {feature_names[spec['feature2']]}")

# Crear features engineered
X_train_engineered, engineered_names = create_engineered_features(
    X_train_scaled, best_features_spec, feature_names
)
X_test_engineered, _ = create_engineered_features(
    X_test_scaled, best_features_spec, feature_names
)

# Combinar
X_train_combined = np.hstack([X_train_scaled, X_train_engineered])
X_test_combined = np.hstack([X_test_scaled, X_test_engineered])

print(f"\nDimensiones:")
print(f"  Original: {X_train_scaled.shape}")
print(f"  Engineered: {X_train_engineered.shape}")
print(f"  Combined: {X_train_combined.shape}")

## 6. Evaluar Mejora

In [ ]:
# Entrenar con features combinadas
final_model = Ridge(alpha=1.0)
final_model.fit(X_train_combined, y_train)

# Predecir
y_pred_baseline = baseline_model.predict(X_test_scaled)
y_pred_engineered = final_model.predict(X_test_combined)

# Métricas
r2_baseline = r2_score(y_test, y_pred_baseline)
r2_engineered = r2_score(y_test, y_pred_engineered)

rmse_baseline = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
rmse_engineered = np.sqrt(mean_squared_error(y_test, y_pred_engineered))

print("="*70)
print("COMPARISON: Original vs Engineered Features")
print("="*70)
print(f"\nBaseline (original features):")
print(f"  R² Score: {r2_baseline:.4f}")
print(f"  RMSE: {rmse_baseline:.2f}")

print(f"\nWith Feature Engineering:")
print(f"  R² Score: {r2_engineered:.4f} ⭐")
print(f"  RMSE: {rmse_engineered:.2f}")

improvement_r2 = ((r2_engineered - r2_baseline) / abs(r2_baseline)) * 100
improvement_rmse = ((rmse_baseline - rmse_engineered) / rmse_baseline) * 100

print(f"\n{'='*70}")
print(f"IMPROVEMENT")
print(f"{'='*70}")
print(f"R² improvement: {improvement_r2:+.2f}%")
print(f"RMSE improvement: {improvement_rmse:+.2f}%")
print(f"New features added: {len(best_features_spec)}")

## 7. Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Fitness evolution
ax = axes[0, 0]
ax.plot(history['best_fitness'], 'g-', linewidth=2, label='Best Fitness')
ax.plot(history['avg_fitness'], 'b--', linewidth=1.5, label='Avg Fitness')
ax.axhline(y=r2_baseline, color='r', linestyle=':', label='Baseline R²')
ax.set_xlabel('Generation')
ax.set_ylabel('Fitness (R² Score)')
ax.set_title('Feature Engineering Evolution')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Number of features
ax = axes[0, 1]
ax.plot(history['n_features'], 'purple', linewidth=2)
ax.set_xlabel('Generation')
ax.set_ylabel('Number of New Features')
ax.set_title('Feature Count Evolution')
ax.grid(True, alpha=0.3)

# 3. Predictions comparison
ax = axes[1, 0]
ax.scatter(y_test, y_pred_baseline, alpha=0.5, label='Baseline', s=30)
ax.scatter(y_test, y_pred_engineered, alpha=0.5, label='Engineered', s=30)
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
ax.set_xlabel('True Values')
ax.set_ylabel('Predictions')
ax.set_title('Predictions: Baseline vs Engineered')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Feature importance
ax = axes[1, 1]
if hasattr(final_model, 'coef_'):
    all_features = list(feature_names) + engineered_names
    importances = np.abs(final_model.coef_)
    
    # Top 10
    top_indices = np.argsort(importances)[-10:][::-1]
    top_features = [all_features[i] for i in top_indices]
    top_importances = importances[top_indices]
    
    ax.barh(range(len(top_features)), top_importances, color='steelblue')
    ax.set_yticks(range(len(top_features)))
    ax.set_yticklabels([f[:30] for f in top_features], fontsize=8)
    ax.set_xlabel('|Coefficient|')
    ax.set_title('Top 10 Most Important Features')
    ax.invert_yaxis()
    ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 🎯 Conclusiones

### ✅ Lo que Logramos

1. **Feature Engineering Automático:**
   - GA exploró espacio de transformaciones
   - Encontró combinaciones útiles automáticamente
   - No necesita conocimiento de dominio

2. **Resultados:**
   - Mejora en R² y RMSE
   - Features interpretables
   - Proceso automatizado

3. **Ventajas vs Feature Engineering Manual:**
   - Más rápido
   - Encuentra interacciones no obvias
   - Escalable a muchas features

### 📚 Lecciones Clave

1. **Transformaciones safe:** Usar operaciones que no generen NaN/Inf
2. **Penalización importante:** Evitar overfitting con muchas features
3. **CV esencial:** Validar que features generalizan
4. **Interpretabilidad:** Mantener features interpretables cuando posible

### 💡 Cuándo Usar

✅ **Bueno para:**
- Datasets tabulares
- Cuando features originales son básicas
- Exploración inicial de interacciones
- Modelos lineales (benefician más de feature engineering)

❌ **Menos útil:**
- Deep learning (aprende features automáticamente)
- Datasets muy grandes (costo computacional)
- Features ya muy engineered

### 🚀 Próximos Pasos

1. Agregar más transformaciones (trigonométricas, estadísticas)
2. Multi-objetivo (accuracy vs complejidad)
3. Combinar con feature selection
4. Probar en tus propios datasets

---

**¡Continúa con más casos de estudio!** 🎉